# 02 — K-Means Clustering: Phân nhóm hành vi coin

**Mục tiêu:** Phân nhóm 10 coin thành **k=3 cụm** dựa trên hành vi thị trường:  
- `return_pct_avg` — lợi nhuận trung bình hàng ngày  
- `volatility_avg` — biên độ giá trung bình (high - low)  
- `volume_usd_avg` — khối lượng giao dịch trung bình (USD)  

**Nguồn dữ liệu:** `gold.fact_market_daily` + `gold.v_weekly_return_by_category`  
**Model:** `sklearn.cluster.KMeans` với k=3  
**Reuse:** Adapt từ `datamining_analysis.py` phần 3 — đổi feature list từ cổ phiếu → crypto

---
**Các bước:**
1. Aggregate đặc trưng trung bình theo coin
2. Chuẩn hoá (StandardScaler)
3. Elbow method chọn k
4. Fit K-Means k=3
5. Visualize clusters (2D scatter, radar chart)
6. Profiling từng cụm


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sqlalchemy import create_engine, text
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

print('Libraries loaded OK')

In [ ]:
# ── Database connection ───────────────────────────────────────────────────────
def load_env(path: str = '../.env') -> dict:
    env = {}
    if not os.path.exists(path):
        path = '../.env.example'
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    k, v = line.split('=', 1)
                    env[k.strip()] = v.strip()
    return env

env  = load_env()
DSN  = (f"postgresql+psycopg2://{env.get('PG_USER','crypto_etl')}"
        f":{env.get('PG_PASSWORD','crypto_etl')}"
        f"@{env.get('PG_HOST','127.0.0.1')}"
        f":{env.get('PG_PORT','5432')}"
        f"/{env.get('PG_DATABASE','crypto_dw_etl')}")
engine = create_engine(DSN)

with engine.connect() as conn:
    n = conn.execute(text('SELECT COUNT(*) FROM gold.fact_market_daily')).scalar()
    print(f'Connected OK — {n:,} rows in gold.fact_market_daily')

In [ ]:
# ── Load & aggregate per-coin features ───────────────────────────────────────
SQL_AGG = """
SELECT
    c.symbol,
    c.full_name,
    cat.category_name,
    cat.risk_level,
    -- Core clustering features
    AVG(f.return_pct)           AS return_pct_avg,
    AVG(f.volatility)           AS volatility_avg,
    AVG(f.volume_usd)           AS volume_usd_avg,
    -- Extra features for profiling
    AVG(f.log_return)           AS log_return_avg,
    AVG(f.average_price)        AS avg_price_usd,
    AVG(f.market_dominance_pct) AS dominance_pct_avg,
    SUM(CASE WHEN f.is_anomaly THEN 1 ELSE 0 END)::FLOAT / COUNT(*) * 100
                                AS anomaly_rate_pct,
    SUM(CASE WHEN f.regime_label = 'Bull' THEN 1 ELSE 0 END)::FLOAT / COUNT(*) * 100
                                AS bull_day_pct,
    SUM(CASE WHEN f.regime_label = 'Bear' THEN 1 ELSE 0 END)::FLOAT / COUNT(*) * 100
                                AS bear_day_pct,
    COUNT(*)                    AS trading_days
FROM gold.fact_market_daily f
JOIN gold.dim_coin c   ON c.coin_id  = f.coin_id
JOIN gold.dim_category cat ON cat.category_id = f.category_id
GROUP BY c.symbol, c.full_name, cat.category_name, cat.risk_level
ORDER BY c.symbol
"""

df_agg = pd.read_sql(SQL_AGG, engine)
print(f'Aggregated: {df_agg.shape[0]} coins × {df_agg.shape[1]} features')
df_agg

## 1. Elbow Method — chọn k tối ưu

In [ ]:
# ── Clustering features ───────────────────────────────────────────────────────
CLUSTER_FEATURES = ['return_pct_avg', 'volatility_avg', 'volume_usd_avg']

X = df_agg[CLUSTER_FEATURES].values
scaler = StandardScaler()
X_s    = scaler.fit_transform(X)

# Elbow method
inertias    = []
silhouettes = []
K_range = range(2, min(len(df_agg), 8))

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_s)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_s, labels))

fig_elbow = make_subplots(rows=1, cols=2,
    subplot_titles=['Elbow Method (Inertia)', 'Silhouette Score'])

fig_elbow.add_trace(go.Scatter(
    x=list(K_range), y=inertias, mode='lines+markers',
    name='Inertia', line=dict(color='#00d4ff', width=2),
    marker=dict(size=8)
), row=1, col=1)

fig_elbow.add_trace(go.Scatter(
    x=list(K_range), y=silhouettes, mode='lines+markers',
    name='Silhouette', line=dict(color='#ff6b6b', width=2),
    marker=dict(size=8)
), row=1, col=2)

# Mark k=3
for col in [1, 2]:
    fig_elbow.add_vline(x=3, line_dash='dot', line_color='#ffd166', row=1, col=col)

fig_elbow.update_layout(template='plotly_dark', height=380,
    title='Chọn số cụm k — Elbow + Silhouette', showlegend=False)
fig_elbow.show()

print(f'Silhouette scores: {dict(zip(K_range, [round(s,3) for s in silhouettes]))}')

## 2. Fit K-Means k=3

In [ ]:
# ── Fit K-Means k=3 ───────────────────────────────────────────────────────────
K = 3
km_final = KMeans(n_clusters=K, random_state=42, n_init=20)
df_agg['cluster'] = km_final.fit_predict(X_s)

# Map cluster → descriptive label based on volatility + return profile
cluster_profiles = (
    df_agg.groupby('cluster')[['return_pct_avg', 'volatility_avg', 'volume_usd_avg']]
    .mean()
    .sort_values('volatility_avg')
)
LABEL_MAP = {}
ordered = cluster_profiles.index.tolist()
LABEL_MAP[ordered[0]] = 'Low Volatility (Stable)'
LABEL_MAP[ordered[1]] = 'Mid Volatility (Growth)'
LABEL_MAP[ordered[2]] = 'High Volatility (Speculative)'
df_agg['cluster_label'] = df_agg['cluster'].map(LABEL_MAP)

print(f'Silhouette (k=3): {silhouette_score(X_s, df_agg["cluster"]):.4f}\n')
print('Cluster assignments:')
print(df_agg[['symbol', 'category_name', 'risk_level', 'cluster_label']].to_string(index=False))

## 3. Visualization

In [ ]:
# ── Plot 1: 2D Scatter — volatility vs return_pct ────────────────────────────
COLOR_MAP = {
    'Low Volatility (Stable)':       '#06d6a0',
    'Mid Volatility (Growth)':       '#ffd166',
    'High Volatility (Speculative)': '#ef476f',
}

fig1 = px.scatter(
    df_agg,
    x='volatility_avg', y='return_pct_avg',
    color='cluster_label',
    size='volume_usd_avg',
    text='symbol',
    color_discrete_map=COLOR_MAP,
    hover_data=['full_name', 'category_name', 'risk_level'],
    title='K-Means Clusters — Volatility vs. Return (bubble = volume)',
    labels={
        'volatility_avg': 'Avg Daily Volatility (High-Low, USD)',
        'return_pct_avg': 'Avg Daily Return (%)',
        'cluster_label': 'Cluster'
    },
    template='plotly_dark', height=520
)
fig1.update_traces(textposition='top center', marker=dict(opacity=0.8, line=dict(width=1, color='white')))
fig1.add_hline(y=0, line_dash='dot', line_color='gray', annotation_text='0% return')
fig1.show()

In [ ]:
# ── Plot 2: PCA 2D view ───────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_s)
df_agg['pca1'] = X_pca[:, 0]
df_agg['pca2'] = X_pca[:, 1]

fig2 = px.scatter(
    df_agg, x='pca1', y='pca2',
    color='cluster_label',
    text='symbol',
    color_discrete_map=COLOR_MAP,
    title=f'PCA 2D Projection — K-Means k=3  (explained var: {pca.explained_variance_ratio_.sum()*100:.1f}%)',
    labels={'pca1': f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
            'pca2': f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)'},
    template='plotly_dark', height=480
)
fig2.update_traces(textposition='top center', marker=dict(size=14, opacity=0.85))
fig2.show()

In [ ]:
# ── Plot 3: Radar chart — cluster profiles ─────────────────────────────────────
RADAR_COLS = ['return_pct_avg', 'volatility_avg', 'dominance_pct_avg',
              'anomaly_rate_pct', 'bull_day_pct']
RADAR_LABELS = ['Avg Return %', 'Avg Volatility', 'Mkt Dominance %',
                'Anomaly Rate %', 'Bull Day %']

# Normalize each feature to [0,1] for radar
df_radar = df_agg.groupby('cluster_label')[RADAR_COLS].mean()
df_radar_norm = (df_radar - df_radar.min()) / (df_radar.max() - df_radar.min() + 1e-9)

fig3 = go.Figure()
colors = list(COLOR_MAP.values())
for i, (label, row) in enumerate(df_radar_norm.iterrows()):
    vals = row.tolist()
    fig3.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=RADAR_LABELS + [RADAR_LABELS[0]],
        fill='toself', name=label,
        line=dict(color=colors[i], width=2),
        fillcolor=colors[i], opacity=0.25
    ))

fig3.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1], showticklabels=False)),
    title='Cluster Profile Radar Chart (normalized)',
    template='plotly_dark', height=500, showlegend=True
)
fig3.show()

In [ ]:
# ── Plot 4: Daily return_pct distribution per cluster ─────────────────────────
SQL_DAILY = """
SELECT c.symbol, f.return_pct, f.regime_label
FROM gold.fact_market_daily f
JOIN gold.dim_coin c ON c.coin_id = f.coin_id
WHERE f.return_pct IS NOT NULL
"""
df_daily = pd.read_sql(SQL_DAILY, engine)
df_daily = df_daily.merge(df_agg[['symbol', 'cluster_label']], on='symbol')

fig4 = px.violin(
    df_daily, x='cluster_label', y='return_pct',
    color='cluster_label', box=True, points=False,
    color_discrete_map=COLOR_MAP,
    title='Daily Return Distribution by Cluster',
    labels={'return_pct': 'Daily Return (%)', 'cluster_label': 'Cluster'},
    template='plotly_dark', height=480
)
fig4.add_hline(y=0, line_dash='dot', line_color='gray')
fig4.update_layout(showlegend=False)
fig4.show()

In [ ]:
# ── Cluster summary table ─────────────────────────────────────────────────────
summary = df_agg.groupby('cluster_label').agg(
    coins            = ('symbol', lambda x: ', '.join(sorted(x))),
    count            = ('symbol', 'count'),
    return_pct_avg   = ('return_pct_avg', lambda x: f'{x.mean():.4f}%'),
    volatility_avg   = ('volatility_avg', lambda x: f'${x.mean():,.2f}'),
    volume_usd_avg   = ('volume_usd_avg', lambda x: f'${x.mean()/1e6:,.1f}M'),
    anomaly_rate_pct = ('anomaly_rate_pct', lambda x: f'{x.mean():.1f}%'),
    bull_day_pct     = ('bull_day_pct', lambda x: f'{x.mean():.1f}%'),
).reset_index()

print('=== Cluster Summary ===')
summary

## 4. Kết luận

| Cluster | Coins | Đặc điểm |
|---------|-------|----------|
| Low Volatility (Stable) | — | Biên độ thấp, return ổn định, volume lớn (BTC, ETH) |
| Mid Volatility (Growth) | — | Cân bằng risk/return, category L1/L2 |
| High Volatility (Speculative) | — | Biên độ cao, nhiều anomaly hơn (Meme coins) |

> **Nhận xét:**  
> - K=3 phù hợp với phân khúc thực tế: Blue-chip (BTC/ETH), Alt-coins, và Speculative.  
> - Silhouette Score đạt mức chấp nhận được với bộ dữ liệu nhỏ (10 coin).  
> - `volume_usd_avg` là feature phân biệt mạnh nhất giữa các cụm.  
> - Kết quả nhất quán với `risk_level` trong `dim_category` (Low/Mid/High).
